# Setup

In [1]:
from statasetup import *


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      18.0
___/   /   /___/   /   /___/       BE—Basic Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-STATA-PC        https://www.stata.com
                                   979-696-4600        stata@stata.com

Stata license: Unlimited-user network, expiring 17 Sep 2026
Serial number: 301909300944
  Licensed to: StataNow/BE 19.5
               StataNow/BE 19.5

Notes:
      1. Unicode is supported; see help unicode_advice.


In [2]:
%%stata
use "Datasets\asg1\PlannedEnv_4countries.dta",clear

# One way ANOVA

## Descriptive Analysis

In [18]:
%%stata
* Do a bit of descriptive analysis (check missing values, coding scheme) by which investigation of the two variable of interest is attempted
codebook country
bysort country: sum behavior if country != ., d
hist behavior, by(country)
list count behavior if behavior >=.


. * Do a bit of descriptive analysis (check missing values, coding scheme) by w
> hich investigation of the two variable of interest is attempted
. codebook country

-------------------------------------------------------------------------------
country                                                             (unlabeled)
-------------------------------------------------------------------------------

                  Type: Numeric (double)
                 Label: countrylabel

                 Range: [1,4]                         Units: 1
         Unique values: 4                         Missing .: 1/801

            Tabulation: Freq.   Numeric  Label
                          199         1  Spain
                          200         2  India
                          200         3  US
                          201         4  Mexico
                            1         .  

. bysort country: sum behavior if country != ., d

-------------------------------------------------------

## Checking for normality

In [23]:
%%stata
* check/verify the distribution central part tallness
sktest behavior if country ==1
sktest behavior if country ==2
sktest behavior if country ==3
sktest behavior if country ==4

bysort country: swilk behavior


. * check/verify the distribution central part tallness
. sktest behavior if country ==1

Skewness and kurtosis tests for normality
                                                         ----- Joint test -----
    Variable |       Obs   Pr(skewness)   Pr(kurtosis)   Adj chi2(2)  Prob>chi2
-------------+-----------------------------------------------------------------
    behavior |       199         0.0408         0.3609          5.06     0.0796

. sktest behavior if country ==2

Skewness and kurtosis tests for normality
                                                         ----- Joint test -----
    Variable |       Obs   Pr(skewness)   Pr(kurtosis)   Adj chi2(2)  Prob>chi2
-------------+-----------------------------------------------------------------
    behavior |       200         0.0053         0.3398          8.00     0.0183

. sktest behavior if country ==3

Skewness and kurtosis tests for normality
                                                         ----- Joint test

## Performing a heterogeneity of variances test

In [6]:
%%stata
robvar(behavior), by(country)


            |         Summary of behavior
    country |        Mean   Std. dev.       Freq.
------------+------------------------------------
      Spain |   2.6669529   .27487569         199
      India |   2.6076151   .27605625         200
         US |   2.2037142   .24876457         200
     Mexico |   2.4197233   .24535415         201
------------+------------------------------------
      Total |   2.4741923   .31773319         800

W0  =  1.6160376   df(3, 796)     Pr > F = 0.18417302

W50 =  1.7191323   df(3, 796)     Pr > F = 0.16155231

W10 =  1.7023315   df(3, 796)     Pr > F = 0.16505076


## Perform One Way ANOVA and determine effect size

In [38]:
%%stata
anova behavior country
estat esize 


. anova behavior country

                         Number of obs =        800    R-squared     =  0.3246
                         Root MSE      =    .261615    Adj R-squared =  0.3220

                  Source | Partial SS         df         MS        F    Prob>F
              -----------+----------------------------------------------------
                   Model |  26.182521          3   8.7275071    127.52  0.0000
                         |
                 country |  26.182521          3   8.7275071    127.52  0.0000
                         |
                Residual |   54.48003        796   .06844225  
              -----------+----------------------------------------------------
                   Total |  80.662551        799   .10095438  

. estat esize 

Effect sizes for linear models

-----------------------------------------------------------------
             Source | Eta-squared     df     [95% conf. interval]
--------------------+-------------------------------------

## Post hoc analysis, on margin means and observed explained variance
1. **Observed means** calculated based on your data: `mean dependent, over(factor)` & `pwmean` & `pwcompare`
2. **Estimated marginal means** obtained by means of linear regression: `margins` & `pwcompare`

In [48]:
%%stata
mean behavior, over(country)
pwmean behavior, over(country) effect
pwmean behavior, over(country) mcompare(tukey) effect


. mean behavior, over(country)

Mean estimation                                  Number of obs = 800

--------------------------------------------------------------------
                   |       Mean   Std. err.     [95% conf. interval]
-------------------+------------------------------------------------
c.behavior@country |
            Spain  |   2.666953   .0194854      2.628704    2.705202
            India  |   2.607615   .0195201      2.569298    2.645932
               US  |   2.203714   .0175903      2.169185    2.238243
           Mexico  |   2.419723   .0173059      2.385753    2.453694
--------------------------------------------------------------------

. pwmean behavior, over(country) effect

Pairwise comparisons of means with equal variances

Over: country

------------------------------------------------------------------------------
             |                            Unadjusted           Unadjusted
    behavior |   Contrast   Std. err.      t    P>|t|     [95%

In [47]:
%%stata
margins country
margins country, pwcompare(effects)
margins country, pwcompare(effects) mcompare(bonferroni)


. margins country

Adjusted predictions                                       Number of obs = 800

Expression: Linear prediction, predict()

------------------------------------------------------------------------------
             |            Delta-method
             |     Margin   std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
     country |
      Spain  |   2.666953   .0185454   143.81   0.000     2.630549    2.703356
      India  |   2.607615    .018499   140.96   0.000     2.571303    2.643928
         US  |   2.203714    .018499   119.13   0.000     2.167402    2.240027
     Mexico  |   2.419723   .0184529   131.13   0.000     2.383501    2.455945
------------------------------------------------------------------------------

. margins country, pwcompare(effects)

Pairwise comparisons of adjusted predictions               Number of obs = 800

Expression: Linear prediction, predict()

----------